# Generate your custom retrieval database

To use this notebook, run ```jupyter notebook``` from the notebook directory.
> ⚠️ **Warning:** You should run this command after setting up a development environment (-dev) as detailed  in the README file.

Install the necessary modules:

In [1]:
import os
from enum import Enum
from IPython.display import display, Markdown
from document_parsing.docling_parser import DoclingParser
from rag.utils import print_dict, get_leaf_classes
from rag.preprocessing.generate_chunks import generate_chunks, AvailableChunkingStrategies
from rag.preprocessing.generate_embeddings import generate_embeddings
from rag.models.embedding_models.embedding_models import EmbeddingModel

## Parse a PDF document

To parse the PDF, run:

In [2]:
parser = DoclingParser()

data_path = os.path.join(os.path.dirname(os.getcwd()), "src", "data")
parsed_document = parser.parse(input_file=os.path.join(data_path, "input_files", "Medical.pdf"),
                               destination_path=os.path.join(data_path, "parsed_files"))

Saved:  eiq-genai-flow_package/retrieval-database-generator/src/data/parsed_files/Medical.md 


Show the parsed document:

In [3]:
display(Markdown('>' + parsed_document.replace('\n', "\n>")))

>
>
>## Guidelines for blood glucose monitoring
>
>- · Always wash and dry your hands carefully before testing your blood glucose.
>- · A new lancet should be used for each test.
>- · Children should check their blood glucose under the supervision of a parent or an adult who understands what the readings mean.
>
>Blood glucose generally needs to be tested 5  -7 times every day before meals (breakfast, lunch and dinner) and before bed. It may be necessary to test more frequently at times, including during the night or when sick.
>
>## Blood glucose monitoring  what is it and why do we do it?
>
>Blood glucose monitoring means checking the blood glucose level to help keep it within the normal range (4-8 mmols/L). This is an essential part of looking after your child's diabetes. Blood glucose levels need to be monitored so that the insulin doses can be adjusted.
>
>## How do we do it and what other equipment is needed?
>
>- 1. We use a glucose meter: this measures blood glucose. You will be shown how to use the meter before you leave hospital.
>- 2. Finger- lancing device: this is needed to prick the   nger for the blood glucose test fi (it is essential to use a new lancet each time you test the blood glucose level).
>- 3. Test strips for the blood glucose and blood ketone meter: A drop of blood goes on the strip to measure blood glucose and/or blood ketone in the meter.
>
>
>
>
>
>## How is the insulin given?
>
>There are three ways insulin can be given:
>
>- 1.  Insulin pen
>- 2.  A syringe and vial of insulin
>- 3.  A pump
>
>## Where do I inject the insulin?
>
>There are three main areas where insulin can be injected:
>
>## Using insulin to treat type 1 diabetes
>
>Insulin is the only way to manage type1 diabetes. Insulin is injected using a short needle into the tissue layer between the skin and the muscle (the subcutaneous tissue). This allows the insulin to be absorbed gradually.
>
>Everyone's lifestyle is different so we will work with you to   nd the best regimen for your child. fi Over time you will learn to adjust the doses for different situations.
>
>Insulin pump therapy is another option for delivering insulin. It is rarely used at diagnosis but may be a suitable option as your child's diabetes journey progresses.
>
>
>
>
>
>
>
>- 1. Abdomen    2.  Legs    3.  Buttock
>
>
>
>## Rotation &amp; care of injection sites
>
>- · Inspect and palpate (touch) injection site for lumps or bruising. If present, avoid injecting into that area until it has resolved.
>- · Rotate injection sites to prevent lumps from occurring. Correct rotation involves spacing insulin injections at least 1cm apart (approx. width of one adult   nger) in the fi same injection zone. Your health care professional will advise you on this.
>- · Injecting through clothes is not a good idea as this may cause an infection at the site and the insulin may not get delivered into the subcutaneous layer and therefore may not work.
>- • Use a new needle for each injection.

## Chunk the parsed document

Available chunking strategies:

In [4]:
for cls_ in AvailableChunkingStrategies:
    print(f" - {cls_.name}")

 - HIRAG
 - SPACY
 - NLTK
 - RECURSIVE
 - FIXED


Set your parameters:

In [5]:
chunking_method = "HIRAG"
chunk_size = 128
chunk_overlap = 64

To chunk the file, run:

In [6]:
chunks = generate_chunks(chunk_size=chunk_size,
                         chunk_overlap=chunk_overlap,
                         files_to_keep=["Medical.md"],
                         chunking_method=AvailableChunkingStrategies[chunking_method])

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Chunking with HiRAG: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [04:22<00:00, 262.96s/it]

Successfully saved Medical_HiRAG_chunks.json at:  eiq-genai-flow_package/retrieval-database-generator/src/data/chunked_files/Medical_HiRAG_chunks.json 


To show the 5 first generated chunks, run:

In [7]:
first_5_dict = dict(list(chunks.items())[:5])
print_dict(first_5_dict)

 - 0:  
     - chunks:  
       ['Always wash and dry your hands carefully.']
     - complete_chunks:   What should you do before testing your blood glucose?;Always wash and dry your hands carefully.
     - question:   What should you do before testing your blood glucose?
     - answer:   Always wash and dry your hands carefully.
 - 1:  
     - chunks:  
       ['Always wash and dry your hands carefully.']
     - complete_chunks:   What should you do before testing your blood glucose?
     - question:   What should you do before testing your blood glucose?
     - answer:   Always wash and dry your hands carefully.
 - 2:  
     - chunks:  
       ['Always wash and dry your hands thoroughly.']
     - complete_chunks:   What is the essential step to take before checking your blood sugar levels?;Always wash and dry your hands thoroughly.
     - question:   What is the essential step to take before checking your blood sugar levels?
     - answer:   Always wash and dry your hands thoroughly.

## Generate RAG database

To generate the database, run:

In [8]:
rag_database = generate_embeddings(
    files_to_keep=["Medical_HiRAG_chunks.json"],  # The file name(s) must correspond to the one generated in the chunking section
    )

Embedding model used: all-MiniLM-L6-v2.onnx 


Enter a brief description of your database content. It will be displayed when loading. (Press Enter to confirm):  Health guide for people with diabetes.


Generating embeddings for Medical_HiRAG_chunks.json file: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 426/426 [00:04<00:00, 103.97it/s]


Successfully saved rag_database.pkl at:  eiq-genai-flow_package/retrieval-database-generator/src/data/rag_database.pkl 
